# 03 — Modeling

Tres experimentos, todos logueados en MLflow bajo el experiment
`otdr_risk_detection`:

1. **Regla de dominio (3dB)** — el "modelo a vencer", ya implementado en
   `weak_labels.label_at_risk`. Se loguea igual para tener un punto de
   comparación objetivo en `04_evaluation`.
2. **No supervisado — IsolationForest** sobre las features numéricas, sin usar
   el weak label para entrenar.
3. **Supervisado contra el weak label** — `RandomForestClassifier`, con
   **split temporal** (no aleatorio, para simular detección temprana real) y
   `class_weight='balanced'` por el desbalance esperado.

Recordatorio: el weak label (`at_risk`) es una regla, no ground truth — ver
`00_business_understanding.ipynb`.


In [ ]:
import sys
from pathlib import Path

SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

import mlflow
import mlflow_utils as MU
from features import MODEL_FEATURE_COLUMNS
import weak_labels as WL

PROCESSED_DIR = Path("../data/processed")
df = pd.read_parquet(PROCESSED_DIR / "features_by_route_timeline.parquet")
df = df.sort_values("TestTime").reset_index(drop=True)

MU.get_or_create_experiment("otdr_risk_detection")
print(f"{len(df)} mediciones cargadas, {df['route_id'].nunique()} rutas.")


## Split temporal

Entrenamos con el 70% más antiguo y evaluamos contra el 30% más reciente — simula el uso real (detectar riesgo en mediciones futuras, no interpolar dentro del mismo período).

In [ ]:
split_idx = int(len(df) * 0.7)
df_train, df_test = df.iloc[:split_idx], df.iloc[split_idx:]

X_train = df_train[MODEL_FEATURE_COLUMNS].fillna(0)
X_test = df_test[MODEL_FEATURE_COLUMNS].fillna(0)
y_train = df_train["at_risk"]
y_test = df_test["at_risk"]

print(f"train={len(df_train)}  test={len(df_test)}  positivos en test={y_test.mean():.1%}")


## (a) Baseline por reglas — punto de comparación

In [ ]:
y_pred_rule = df_test["at_risk"]  # la regla ES la etiqueta, por definición

metrics_rule = {
    "precision": precision_score(y_test, y_pred_rule, zero_division=0),
    "recall": recall_score(y_test, y_pred_rule, zero_division=0),
    "f1": f1_score(y_test, y_pred_rule, zero_division=0),
}

run_id_rule = MU.log_model_run(
    run_name="rule_based_baseline",
    params={"threshold_db": WL.DEFAULT_THRESHOLD_DB, "margin_db": WL.DEFAULT_MARGIN_DB},
    metrics=metrics_rule,
)
print(metrics_rule)


## (b) No supervisado — IsolationForest

In [ ]:
iso_params = {"n_estimators": 200, "contamination": 0.1, "random_state": 42}
iso = IsolationForest(**iso_params).fit(X_train)

# IsolationForest: -1 = anomalía. Se mapea a 1 (at_risk) para comparar con el weak label.
y_pred_iso = (iso.predict(X_test) == -1).astype(int)

metrics_iso = {
    "precision": precision_score(y_test, y_pred_iso, zero_division=0),
    "recall": recall_score(y_test, y_pred_iso, zero_division=0),
    "f1": f1_score(y_test, y_pred_iso, zero_division=0),
}

run_id_iso = MU.log_model_run(
    run_name="isolation_forest",
    params=iso_params,
    metrics=metrics_iso,
    model=iso,
)
print(metrics_iso)


## (c) Supervisado contra el weak label — RandomForestClassifier

In [ ]:
rf_params = {"n_estimators": 200, "max_depth": 6, "class_weight": "balanced", "random_state": 42}
rf = RandomForestClassifier(**rf_params).fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

metrics_rf = {
    "precision": precision_score(y_test, y_pred_rf, zero_division=0),
    "recall": recall_score(y_test, y_pred_rf, zero_division=0),
    "f1": f1_score(y_test, y_pred_rf, zero_division=0),
}

run_id_rf = MU.log_model_run(
    run_name="random_forest_supervised",
    params=rf_params,
    metrics=metrics_rf,
    model=rf,
)
print(metrics_rf)
print(confusion_matrix(y_test, y_pred_rf))


Los `run_id` de los 3 experimentos (`run_id_rule` no tiene modelo logueado, solo métricas; `run_id_iso` y `run_id_rf` sí) se retoman en `04_evaluation.ipynb` para comparar y registrar el mejor en el Model Registry.